<a href="https://colab.research.google.com/github/Joangeek/Card-product.github.io/blob/main/Gemma4_Colab_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔷 Gemma 4 — Entorno Completo para Google Colab (v2)

> **Compatible con todos los modelos Gemma 4:** `E2B`, `E4B`, `26B-A4B (MoE)`, `31B`

| Modelo | Parámetros efectivos | VRAM mínima | Capacidades |
|--------|---------------------|-------------|-------------|
| `gemma-4-e2b-it` | 2B | ~6 GB | Texto, Imagen, Audio |
| `gemma-4-e4b-it` | 4B | ~10 GB | Texto, Imagen, Audio |
| `gemma-4-26b-a4b-it` | 26B (activa 4B) | ~16 GB | Texto, Imagen |
| `gemma-4-31b-it` | 31B | ~22 GB (4-bit) | Texto, Imagen |

### ⚙️ Antes de empezar
1. Ve a **Runtime → Change runtime type** y selecciona **T4 GPU**
2. Ejecuta las celdas en orden de arriba a abajo
3. **NO saltes la celda 1** — es obligatoria cada vez que inicies el notebook


## 📦 Celda 1 — Instalación de dependencias
> ⚠️ Ejecuta esta celda SIEMPRE primero. Luego **reinicia la sesión** antes de continuar.

In [ ]:
!pip install -q "numpy==1.26.4" --force-reinstall
!pip install -q "Pillow==10.4.0" --force-reinstall
!pip install -q transformers --upgrade
!pip install -q -U bitsandbytes>=0.46.1
!pip install -q accelerate sentencepiece huggingface_hub
!pip install -q --no-deps unsloth unsloth_zoo peft trl
!pip install -q soundfile

print("\n✅ Instalación completa.")
print("⚠️  Ahora ve a: Entorno de ejecución → Reiniciar sesión")
print("   Después del reinicio, ejecuta desde la Celda 2 en adelante.")


## ✅ Celda 2 — Verificar versiones
> Ejecuta esto después de reiniciar la sesión para confirmar que todo está bien.

In [ ]:
import numpy as np
import PIL
import torch
import transformers

print("📊 Versiones instaladas:")
print(f"   numpy        : {np.__version__}")
print(f"   Pillow       : {PIL.__version__}")
print(f"   torch        : {torch.__version__}")
print(f"   transformers : {transformers.__version__}")
print(f"   CUDA         : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"   GPU          : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM total   : {vram:.1f} GB")
    if vram >= 15:
        print("   → Puedes correr: E2B, E4B, 26B-A4B (4-bit)")
    elif vram >= 10:
        print("   → Puedes correr: E2B, E4B")
    else:
        print("   → Puedes correr: E2B")

print("\n✅ Todo listo, continúa con la Celda 3.")


## 🔑 Celda 3 — Autenticación en Hugging Face

1. Ve a [huggingface.co/google/gemma-4-e4b-it](https://huggingface.co/google/gemma-4-e4b-it) y acepta los términos
2. Genera un token en **Settings → Access Tokens → New Token** (tipo: Read)
3. En Colab ve a **🔑 Secrets** y agrega tu token con el nombre `HF_TOKEN`


In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print("✅ Autenticado con HF_TOKEN desde Secrets.")
except Exception:
    print("Secrets no encontrado. Ingresa tu token manualmente:")
    login()


## 🎛️ Celda 4 — Selector de modelo
> Cambia `MODEL_CHOICE` según tu GPU. Con T4 (16GB) se recomienda `e4b`.

In [ ]:
# ─────────────────────────────────────────────────
# ✏️  CAMBIA AQUÍ el modelo que quieres usar
# ─────────────────────────────────────────────────
MODEL_CHOICE = "e4b"   # Opciones: "e2b" | "e4b" | "26b" | "31b"
LOAD_IN_4BIT = True    # True = menos VRAM, recomendado para T4
# ─────────────────────────────────────────────────

MODEL_MAP = {
    "e2b":  {"hf_id": "google/gemma-4-e2b-it",     "vram": "~6 GB",  "notas": "Texto + Imagen + Audio"},
    "e4b":  {"hf_id": "google/gemma-4-e4b-it",     "vram": "~10 GB", "notas": "Texto + Imagen + Audio"},
    "26b":  {"hf_id": "google/gemma-4-26b-a4b-it", "vram": "~16 GB", "notas": "MoE, Texto + Imagen"},
    "31b":  {"hf_id": "google/gemma-4-31b-it",     "vram": "~22 GB", "notas": "Dense, Texto + Imagen"},
}

selected  = MODEL_MAP[MODEL_CHOICE]
MODEL_ID  = selected["hf_id"]

print(f"📌 Modelo seleccionado : {MODEL_ID}")
print(f"   VRAM estimada       : {selected['vram']}")
print(f"   Capacidades         : {selected['notas']}")
print(f"   Cuantización 4-bit  : {LOAD_IN_4BIT}")


## 🚀 Celda 5 — Cargar modelo y procesador
> ⏱️ Tarda 5-15 minutos descargando el modelo. Es normal que parezca congelado.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig

print(f"⏳ Cargando {MODEL_ID} ...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=LOAD_IN_4BIT,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
) if LOAD_IN_4BIT else None

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16 if not LOAD_IN_4BIT else None,
    attn_implementation="eager",
)

model.eval()
print("\n✅ Modelo cargado exitosamente.")
if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"   VRAM usada: {used:.1f} / {total:.1f} GB ({used/total*100:.0f}%)")


## 💬 Celda 6 — Chat de texto

In [ ]:
def chat(messages, max_new_tokens=512, temperature=0.7):
    """
    Envía mensajes al modelo y retorna la respuesta.
    Roles válidos: 'system', 'user', 'assistant'
    """
    # Convertir contenido string a formato lista (requerido por transformers >= 5.6)
    formatted = []
    for m in messages:
        if isinstance(m["content"], str):
            formatted.append({
                "role": m["role"],
                "content": [{"type": "text", "text": m["content"]}]
            })
        else:
            formatted.append(m)

    inputs = processor.apply_chat_template(
        formatted,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            top_p=0.95,
        )

    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return processor.decode(new_tokens, skip_special_tokens=True)


# ── Prueba rápida ──
respuesta = chat([
    {"role": "system", "content": "Eres un asistente experto en inteligencia artificial."},
    {"role": "user",   "content": "¿Qué hace especial a Gemma 4 respecto a versiones anteriores?"}
])

print("🤖 Gemma 4 responde:\n")
print(respuesta)


## 🖼️ Celda 7 — Chat con imágenes

In [ ]:
from PIL import Image
import requests
from io import BytesIO

def chat_con_imagen(prompt_texto, url_imagen=None, imagen_local=None, max_new_tokens=512):
    """
    Envía imagen + texto al modelo.
    Proporciona url_imagen o imagen_local (ruta de archivo).
    """
    if url_imagen:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url_imagen, headers=headers, timeout=10)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    elif imagen_local:
        image = Image.open(imagen_local).convert("RGB")
    else:
        raise ValueError("Proporciona url_imagen o imagen_local.")

    # Gemma 4: imagen primero, texto después
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text": prompt_texto}
        ]
    }]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)

    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return processor.decode(new_tokens, skip_special_tokens=True)


# ── Prueba con imagen pública ──
# Puedes cambiar esta URL por cualquier imagen accesible
URL_IMAGEN = "https://upload.wikimedia.org/wikipedia/commons/a/a7/Camponotus_flavomarginatus_ant.jpg"

respuesta = chat_con_imagen(
    prompt_texto="Describe detalladamente lo que ves en esta imagen.",
    url_imagen=URL_IMAGEN
)
print("🖼️ Respuesta con imagen:\n")
print(respuesta)


## 🎙️ Celda 8 — Chat con audio (solo E2B y E4B)

In [ ]:
if MODEL_CHOICE not in ["e2b", "e4b"]:
    print(f"⚠️ El modelo '{MODEL_CHOICE}' no soporta audio. Usa 'e2b' o 'e4b'.")
else:
    import soundfile as sf

    def chat_con_audio(prompt_texto, ruta_audio, max_new_tokens=512):
        """
        Procesa audio + texto.
        El archivo debe ser WAV, máximo 30 segundos.
        Para subir tu archivo: panel izquierdo → ícono carpeta → subir archivo
        Luego usa ruta_audio='/content/tu_archivo.wav'
        """
        audio_data, sample_rate = sf.read(ruta_audio)
        if audio_data.ndim > 1:
            audio_data = audio_data.mean(axis=1)  # convertir a mono

        messages = [{
            "role": "user",
            "content": [
                {"type": "audio", "audio": audio_data, "sampling_rate": sample_rate},
                {"type": "text",  "text": prompt_texto}
            ]
        }]

        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_tensors="pt",
            return_dict=True,
        ).to(model.device)

        with torch.inference_mode():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)

        new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
        return processor.decode(new_tokens, skip_special_tokens=True)

    print("✅ Función chat_con_audio() lista.")
    print("   Uso: chat_con_audio('¿Qué dice?', '/content/tu_archivo.wav')")
    print("\n   Para subir tu archivo WAV:")
    print("   1. Clic en ícono de carpeta 📁 en panel izquierdo")
    print("   2. Clic en ícono subir archivo ⬆️")
    print("   3. Selecciona tu archivo .wav")


## 🎙️ Celda 9 — Probar audio con archivo subido
> Primero sube tu archivo WAV usando el panel de archivos de Colab (📁)

In [ ]:
# ✏️ Cambia el nombre del archivo por el tuyo
ARCHIVO_AUDIO = "/content/tu_archivo.wav"

respuesta = chat_con_audio(
    prompt_texto="Describe o transcribe este audio.",
    ruta_audio=ARCHIVO_AUDIO
)
print("🎙️ Gemma 4 responde:\n")
print(respuesta)


## 🧠 Celda 10 — Modo Thinking (razonamiento paso a paso)

In [ ]:
problema = """
Un tren sale de la ciudad A a 60 km/h. Otro tren sale de la ciudad B
(a 300 km de distancia) a 90 km/h en dirección contraria.
¿En cuánto tiempo se encontrarán?
"""

respuesta = chat(
    messages=[
        {"role": "system", "content": "Eres un experto en matemáticas. Piensa paso a paso antes de responder."},
        {"role": "user",   "content": problema}
    ],
    max_new_tokens=1024,
    temperature=0.3
)

print("🧠 Respuesta con razonamiento:\n")
print(respuesta)


## 🔁 Celda 11 — Chat multi-turno interactivo

In [ ]:
def chat_interactivo(system_prompt="Eres un asistente útil en español."):
    """
    Conversación multi-turno en la terminal.
    Escribe 'salir' para terminar.
    Escribe 'limpiar' para reiniciar el historial.
    """
    historial = [{"role": "system", "content": system_prompt}]
    print(f"🤖 Gemma 4 ({MODEL_CHOICE}) — Chat interactivo")
    print(f"   System: {system_prompt}")
    print("   Escribe 'salir' para terminar, 'limpiar' para reiniciar.\n")

    while True:
        try:
            entrada = input("👤 Tú: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 Sesión terminada.")
            break

        if entrada.lower() in ["salir", "exit", "quit"]:
            print("👋 ¡Hasta luego!")
            break
        if entrada.lower() == "limpiar":
            historial = [{"role": "system", "content": system_prompt}]
            print("🔄 Historial limpiado.\n")
            continue
        if not entrada:
            continue

        historial.append({"role": "user", "content": entrada})
        respuesta = chat(historial, max_new_tokens=512)
        historial.append({"role": "assistant", "content": respuesta})
        print(f"\n🤖 Gemma 4: {respuesta}\n")


# ✏️ Cambia el system prompt por el que quieras
chat_interactivo("Eres un asistente experto en ciencia de datos en español.")


## 📊 Celda 12 — Resumen del entorno

In [ ]:
import torch
import transformers
import PIL
import numpy as np

print("=" * 55)
print("📊 RESUMEN DEL ENTORNO GEMMA 4")
print("=" * 55)
print(f"  Modelo activo    : {MODEL_ID}")
print(f"  Cuantización     : {'4-bit (bitsandbytes)' if LOAD_IN_4BIT else 'bfloat16'}")
print(f"  transformers     : {transformers.__version__}")
print(f"  PyTorch          : {torch.__version__}")
print(f"  Pillow           : {PIL.__version__}")
print(f"  numpy            : {np.__version__}")
print(f"  CUDA disponible  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU              : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM usada       : {used:.1f} / {total:.1f} GB ({used/total*100:.0f}%)")
print("=" * 55)
print("\n🚀 Funciones disponibles:")
print("  chat(messages)                    → texto")
print("  chat_con_imagen(prompt, url)      → texto + imagen")
if MODEL_CHOICE in ['e2b', 'e4b']:
    print("  chat_con_audio(prompt, ruta)      → texto + audio")
print("  chat_interactivo(system_prompt)   → multi-turno")
print("\n✅ Todo funcionando correctamente.")
